<a href="https://colab.research.google.com/github/CodeCanvas-Ak/Ankpro.github.io/blob/master/python-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# 1. 下载官方数据集
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

# 2. 导入库
import torch
import torch.nn.functional as F

# 3. 读取数据（文件名改对）
words = open("names.txt", "r").read().splitlines()

# 4. 构建字符映射（修正 itos）
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi["."] = 0

itos = {i:s for s,i in stoi.items()}  # ✅ 修正这里
vocab_size = len(stoi)

# 5. 构造 trigram 数据
xs = []
ys = []

for w in words:
    chs = [".", "."] + list(w) + ["."]
    for i in range(len(w) + 1):
        x1 = stoi[chs[i]]
        x2 = stoi[chs[i+1]]
        y  = stoi[chs[i+2]]
        xs.append([x1, x2])
        ys.append(y)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

# 6. 统计三元组次数
N = torch.zeros((vocab_size, vocab_size, vocab_size), dtype=torch.int32)

for x,y in zip(xs, ys):
    x1, x2 = x.tolist()
    N[x1, x2, y] += 1

# 7. 转概率 + 平滑
P = (N + 1).float()
P = P / P.sum(dim=2, keepdim=True)

# 8. 计算损失（修正索引）
loss = -torch.log(P[xs[:,0], xs[:,1], ys]).mean()
print("trigram loss =", loss.item())

--2026-04-21 14:04:51--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt.4’

names.txt.4         100%[===================>] 222.80K  --.-KB/s    in 0.003s  

2026-04-21 14:04:51 (64.1 MB/s) - ‘names.txt.4’ saved [228145/228145]

trigram loss = 2.212582588195801
